# Fine-tuning Facebook's Vision Transformer (ViT) on Old Hebrew Script-Mode Images


## Environment setup


In [15]:
!pip install --upgrade "transformers[torch]" datasets accelerate timm torchvision evaluate pandas matplotlib scikit-learn tensorboard safetensors --quiet


## Imports & global configuration

In [4]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

from pathlib import Path
import numpy as np
import pandas as pd 
import random
import matplotlib.pyplot as plt
import json

from collections import Counter, defaultdict
from typing import Optional, Dict, Any

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.utils.data import Dataset 
from tqdm.auto import tqdm

from PIL import Image
from transformers import TrainingArguments, Trainer, AutoImageProcessor, AutoModel
import evaluate
from safetensors.torch import load_file
import re

# Ensure we have a GPU (falls back gracefully to CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


/home/suliman/miniconda3/envs/transformer_class/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


## Define paths & label mappings

In [5]:
BASE_DIR = Path("ASC_dataset_v3_augmented_patchified")
TRAIN_DIR = BASE_DIR / "train"
VAL_DIR   = BASE_DIR / "val"
TEST_DIR  = BASE_DIR / "test"

RESULTS_DIR = Path("mil_vit_results")
os.makedirs(RESULTS_DIR, exist_ok=True)

labels = ["french", "german"]
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}

print("Label mapping:", label2id)


Label mapping: {'french': 0, 'german': 1}


## Define the Models

In [6]:

class AttentionMILHead(nn.Module):
    """
    Attention-based MIL head with dropout regularization.
    H: (N_i, d) patch embeddings for a single bag.
    """
    def __init__(self, d_model: int, num_labels: int, dropout: float = 0.3):
        super().__init__()
        self.instance_dropout = nn.Dropout(dropout)
        self.att_fc = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.Tanh(),
            nn.Linear(d_model, 1),
        )
        self.bag_dropout = nn.Dropout(dropout)
        self.cls_fc = nn.Linear(d_model, num_labels)

    def forward(self, H: torch.Tensor):
        # H: (N_i, d)
        H = self.instance_dropout(H)              # drop some patch embeddings
        a = self.att_fc(H)                        # (N_i, 1)
        a = torch.softmax(a, dim=0)               # attention over instances
        M = (a * H).sum(dim=0)                    # (d,) bag embedding
        M = self.bag_dropout(M)                   # dropout on bag representation
        logits = self.cls_fc(M)                   # (num_labels,)
        return logits.unsqueeze(0), a             # (1, num_labels), (N_i, 1)


class VisionTransformerMILModel(nn.Module):
    """
    Generic ViT/DeiT backbone + attention-based MIL head for page-level classification.

    Expected input (from collator):
        pixel_values: (N_total, 3, 224, 224)
        bag_indices:  (N_total,) long, values in [0, num_bags_in_batch-1]
        labels:       (num_bags_in_batch,) long
    """
    def __init__(self, model_name: str, num_labels: int = 2, mil_dropout: float = 0.3):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        d_model = self.backbone.config.hidden_size
        self.mil_head = AttentionMILHead(d_model, num_labels, dropout=mil_dropout)
        self.num_labels = num_labels

    def forward(
        self,
        pixel_values: torch.Tensor,
        bag_indices: torch.Tensor,
        labels: Optional[torch.Tensor] = None,
    ) -> Dict[str, Any]:
        outputs = self.backbone(pixel_values=pixel_values)
        cls_embeddings = outputs.last_hidden_state[:, 0, :]  # (N_total, d)

        unique_bags = torch.unique(bag_indices)
        bag_logits_list = []

        for b in unique_bags:
            mask = (bag_indices == b)
            H_bag = cls_embeddings[mask]              # (N_i, d)
            logits_b, _ = self.mil_head(H_bag)
            bag_logits_list.append(logits_b)

        logits = torch.cat(bag_logits_list, dim=0)    # (num_bags, num_labels)

        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits, labels)

        return {"loss": loss, "logits": logits}



## Prepare the Datasets

In [7]:


class PageBagDataset(Dataset):
    """
    One item = one page (bag).
    Each item returns:
        {
          "images": [PIL.Image, ...],
          "labels": int
        }
    """
    def __init__(self, bags):
        """
        bags: list of dicts
            {
                "patch_paths": [str, ...],
                "label": int
            }
        """
        self.bags = bags

    def __len__(self):
        return len(self.bags)

    def __getitem__(self, idx):
        bag = self.bags[idx]
        paths = bag["patch_paths"]
        label = bag["label"]

        images = [Image.open(p).convert("RGB") for p in paths]
        return {"images": images, "labels": label}


In [8]:



def build_bags(split_dir: Path):
    """
    Create MIL bags from a split directory structured as:
        split_dir/
            french/
            german/

    Filenames follow:
        {ms_id}_{page_id}_{region_id}_y{...}_x{...}_{patch_idx}.png
        e.g. 026_003_00_y03808_x01344_0125.png

    We group by page = (ms_id, page_id), i.e. all regions on the same page
    belong to the same bag.

    Each bag is:
        {
            "patch_paths": [list of image paths],
            "label": 0 or 1
        }
    """
    bags_dict = defaultdict(lambda: {"patch_paths": [], "label": None})

    for script_label in ["french", "german"]:
        class_dir = split_dir / script_label
        if not class_dir.exists():
            continue

        for fname in os.listdir(class_dir):
            if not fname.lower().endswith((".png", ".jpg", ".jpeg")):
                continue

            fpath = class_dir / fname

            # Strip extension
            stem = os.path.splitext(fname)[0]  # e.g. "026_003_00_y03808_x01344_0125"
            parts = stem.split("_")

            if len(parts) < 3:
                raise ValueError(f"Unexpected filename format: {fname}")

            ms_id    = parts[0]  # "026"
            page_id  = parts[1]  # "003"
            # region_id = parts[2]  # "00" (text region) - we ignore for page grouping

            # Page-level key (ms_id + page_id)
            page_key = f"{ms_id}_{page_id}"

            # Include script_label to avoid collisions between splits/classes
            bag_key = f"{script_label}_{page_key}"

            bags_dict[bag_key]["patch_paths"].append(str(fpath))
            bags_dict[bag_key]["label"] = label2id[script_label]

    return list(bags_dict.values())


# Build bags for all splits
train_bags = build_bags(TRAIN_DIR)
val_bags   = build_bags(VAL_DIR)
test_bags  = build_bags(TEST_DIR)

print(f"Train bags (pages): {len(train_bags)}")
print(f"Val bags   (pages): {len(val_bags)}")
print(f"Test bags  (pages): {len(test_bags)}")

# Quick sanity check: first bag
print(train_bags[0]["label"], len(train_bags[0]["patch_paths"]), "patches")
print(train_bags[0]["patch_paths"][0])


Train bags (pages): 101
Val bags   (pages): 16
Test bags  (pages): 25
0 685 patches
ASC_dataset_v3_augmented_patchified/train/french/029_000_00_masked_inv_gray_y00000_x00448_0002.png


In [9]:

class MILDataCollator:
    """
    Collator for MIL:
        Input batch: list of items from PageBagDataset
        Output:
            pixel_values: (N_total, 3, 224, 224)
            bag_indices:  (N_total,)
            labels:       (num_bags,)
    """
    def __init__(self, processor, max_patches_per_bag: int = 256, shuffle_patches: bool = True):
        self.processor = processor
        self.max_patches_per_bag = max_patches_per_bag
        self.shuffle_patches = shuffle_patches

    def __call__(self, batch):
        all_pixel_values = []
        all_bag_indices = []
        labels = []

        for bag_idx, item in enumerate(batch):
            images = item["images"]  # list of PIL images
            label = item["labels"]

            # ---- Subsample patches per bag to control memory ----
            n = len(images)
            if n > self.max_patches_per_bag:
                if self.shuffle_patches:
                    idxs = random.sample(range(n), self.max_patches_per_bag)
                else:
                    idxs = list(range(self.max_patches_per_bag))
                images = [images[i] for i in idxs]

            inputs = self.processor(
                images=images,
                return_tensors="pt"
            )
            pixel_values = inputs["pixel_values"]  # (N_i, 3, 224, 224)

            all_pixel_values.append(pixel_values)
            all_bag_indices.append(
                torch.full(
                    (pixel_values.size(0),),
                    bag_idx,
                    dtype=torch.long,
                )
            )
            labels.append(label)

        pixel_values = torch.cat(all_pixel_values, dim=0)  # (N_total, 3, 224, 224)
        bag_indices = torch.cat(all_bag_indices, dim=0)    # (N_total,)
        labels = torch.tensor(labels, dtype=torch.long)    # (num_bags,)

        return {
            "pixel_values": pixel_values,
            "bag_indices": bag_indices,
            "labels": labels,
        }


## Prepare Model for Training

In [10]:

model_name = "facebook/deit-small-patch16-224"

# 1) Image processor (disable resize/crop if patches are already 224x224)
processor = AutoImageProcessor.from_pretrained(
    model_name,
    do_resize=False,
    do_center_crop=False,
)

# 2) Build MIL datasets
train_dataset = PageBagDataset(train_bags)
valid_dataset = PageBagDataset(val_bags)

# 3) Model
model = VisionTransformerMILModel(model_name=model_name, num_labels=2)

# 4) Data collator
data_collator = MILDataCollator(processor)

# 5) Metrics (page-level)
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=preds, references=labels)

# 6) Training arguments
training_args = TrainingArguments(
    output_dir=RESULTS_DIR,
    remove_unused_columns=False,
    eval_strategy="epoch",   # fine to keep, but we won't rely on it
    save_strategy="epoch",         # save checkpoint every epoch
    load_best_model_at_end=False,  # we'll choose best checkpoint ourselves
    num_train_epochs=6,
    per_device_train_batch_size=1, # 1 bag/page per step
    per_device_eval_batch_size=1,
    learning_rate=2e-5,
    weight_decay=0.01,
    fp16=True,
    logging_steps=50,
    report_to="tensorboard",
    # optional: to keep all checkpoints
    save_total_limit=None,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.
Some weights of ViTModel were not initialized from the model checkpoint at facebook/deit-small-patch16-224 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


## Fine Tune!

In [11]:
trainer.train()


Epoch,Training Loss,Validation Loss,Accuracy
1,1.332400,0.588394,0.562500
2,0.797900,0.303145,0.875000
3,0.358800,0.891894,0.750000
4,0.742100,0.375538,0.812500
5,0.386300,0.291863,0.812500
6,0.097900,0.157827,0.937500


TrainOutput(global_step=606, training_loss=0.6843916268238533, metrics={'train_runtime': 1130.2968, 'train_samples_per_second': 0.536, 'train_steps_per_second': 0.536, 'total_flos': 0.0, 'train_loss': 0.6843916268238533, 'epoch': 6.0})

## Evaluate on validation set

In [11]:


def extract_page_id_from_path(path_str: str) -> str:
    """
    From: 026_003_00_y03808_x01344_0125.png
    To:   "026_003"
    """
    fname = Path(path_str).name
    stem = fname.split(".")[0]
    parts = stem.split("_")
    if len(parts) < 2:
        return stem
    return f"{parts[0]}_{parts[1]}"


@torch.no_grad()
def infer_bag_full(bag, model, processor, device, chunk_size: int = 256):
    """
    Compute logits for ONE bag (page) using ALL patches.
    """
    paths = bag["patch_paths"]
    all_cls_cpu = []

    for start in range(0, len(paths), chunk_size):
        chunk_paths = paths[start : start + chunk_size]
        images = [Image.open(p).convert("RGB") for p in chunk_paths]

        inputs = processor(images=images, return_tensors="pt")
        pixel_values = inputs["pixel_values"].to(device)

        outputs = model.backbone(pixel_values=pixel_values)
        cls_emb = outputs.last_hidden_state[:, 0, :]  # (chunk, d)

        all_cls_cpu.append(cls_emb.cpu())

        del pixel_values, outputs, cls_emb
        torch.cuda.empty_cache()

    H = torch.cat(all_cls_cpu, dim=0).to(device)  # (N_patches, d)
    logits, _ = model.mil_head(H)                 # (1, num_labels)
    return logits.squeeze(0)                      # (num_labels,)


@torch.no_grad()
def evaluate_full_bag_on_split_thesis(
    bags,
    model,
    processor,
    device,
    chunk_size: int = 256,
    label_map=None,                 # {0:"French", 1:"German"} (string case for thesis)
    sort_by=("manuscript_id", "page_num"),
    round_probs=3,
    add_correct_col=True,
):
    """
    Returns:
        acc (float), avg_loss (float),
        df_all (full debug dataframe),
        df_thesis (clean thesis-ready dataframe)
    """
    if label_map is None:
        label_map = {0: "French", 1: "German"}

    # ---- collect per-page info ----
    rows = []
    model.to(device)
    model.eval()

    for bag in tqdm(bags, desc="Full-bag eval"):
        patch_paths = bag["patch_paths"]
        page_id = extract_page_id_from_path(patch_paths[0])
        num_patches = len(patch_paths)

        logits = infer_bag_full(bag, model, processor, device, chunk_size=chunk_size)
        logits_np = logits.detach().cpu().numpy()

        rows.append({
            "page_id": page_id,
            "true_label_int": int(bag["label"]),
            "num_patches": int(num_patches),
            "logit_french": float(logits_np[0]),
            "logit_german": float(logits_np[1]),
        })

    df_all = pd.DataFrame(rows)

    # ---- compute probs/preds ----
    logits_arr = df_all[["logit_french", "logit_german"]].to_numpy()
    labels_arr = df_all["true_label_int"].to_numpy(dtype=int)

    # loss over pages
    avg_loss = F.cross_entropy(
        torch.from_numpy(logits_arr),
        torch.from_numpy(labels_arr)
    ).item()

    # numerically-stable softmax
    exp_logits = np.exp(logits_arr - logits_arr.max(axis=1, keepdims=True))
    probs = exp_logits / exp_logits.sum(axis=1, keepdims=True)
    preds = probs.argmax(axis=1)
    acc = (preds == labels_arr).mean()

    df_all["prob_french"] = probs[:, 0]
    df_all["prob_german"] = probs[:, 1]
    df_all["pred_label_int"] = preds

    # string labels for reporting
    df_all["true_label"] = df_all["true_label_int"].map(label_map)
    df_all["pred_label"] = df_all["pred_label_int"].map(label_map)

    # ---- optional: split page_id into manuscript/page numeric columns for better sorting ----
    # page_id like "026_003" -> manuscript_id="026", page_num=3
    tmp = df_all["page_id"].str.split("_", expand=True)
    if tmp.shape[1] >= 2:
        df_all["manuscript_id"] = tmp[0]
        # keep leading zeros for display, but store numeric for sorting
        df_all["page_num"] = pd.to_numeric(tmp[1], errors="coerce")
    else:
        df_all["manuscript_id"] = df_all["page_id"]
        df_all["page_num"] = np.nan

    if add_correct_col:
        df_all["correct"] = (df_all["pred_label_int"] == df_all["true_label_int"])

    # ---- thesis-facing table: keep only what you want to show ----
    df_thesis = df_all[[
        "page_id",
        "true_label",
        "pred_label",
        "prob_french",
        "prob_german",
        "num_patches",
        "manuscript_id",
        "page_num",
    ]].copy()

    # rounding / formatting for nicer tables
    df_thesis["prob_french"] = df_thesis["prob_french"].round(round_probs)
    df_thesis["prob_german"] = df_thesis["prob_german"].round(round_probs)

    # sort for readability
    for col in sort_by:
        if col not in df_thesis.columns:
            raise ValueError(f"sort_by column '{col}' not in df_thesis columns")
    df_thesis = df_thesis.sort_values(list(sort_by), kind="mergesort").reset_index(drop=True)

    # drop helper sort columns from final print table (optional)
    df_thesis = df_thesis.drop(columns=["manuscript_id", "page_num"])
    return acc, avg_loss, df_all, df_thesis


In [12]:

# ----------------------------
# Config
# ----------------------------
EPS = 1e-6
CHUNK_SIZE = 256

def ckpt_step(p: Path) -> int:
    # checkpoint-1234 -> 1234 (fallback: 0)
    m = re.search(r"checkpoint-(\d+)$", p.name)
    return int(m.group(1)) if m else 0

def load_checkpoint_weights(model, ckpt_dir: Path):
    """
    Tries common formats in priority order:
      1) model.safetensors
      2) pytorch_model.bin
      3) model.bin
    """
    candidates = [
        ckpt_dir / "model.safetensors",
        ckpt_dir / "pytorch_model.bin",
        ckpt_dir / "model.bin",
    ]
    for wpath in candidates:
        if wpath.exists():
            if wpath.suffix == ".safetensors":
                state_dict = load_file(wpath)
            else:
                state_dict = torch.load(wpath, map_location="cpu")
            missing, unexpected = model.load_state_dict(state_dict, strict=False)
            return str(wpath), missing, unexpected
    raise FileNotFoundError(f"No checkpoint weights found in: {ckpt_dir}")

# ----------------------------
# Find checkpoints
# ----------------------------
ckpt_dirs = sorted(
    [p for p in RESULTS_DIR.glob("checkpoint-*") if p.is_dir()],
    key=ckpt_step,
)

print("Found checkpoints:", [d.name for d in ckpt_dirs])

best_ckpt = None
best_acc = -1.0
best_loss = float("inf")

val_results_per_ckpt = {}

# Optional: keep the best checkpoint's df for later reporting
best_val_df_thesis = None  # if you switch evaluate_* to return thesis df
best_val_df_raw = None

for ckpt_dir in ckpt_dirs:
    print(f"\nEvaluating full-bag val on {ckpt_dir.name} ...")

    # Rebuild model exactly as in training
    model = VisionTransformerMILModel(
        model_name=model_name,
        num_labels=2,
        mil_dropout=0.3,
    ).to(device)

    # Load weights (robust)
    wpath, missing, unexpected = load_checkpoint_weights(model, ckpt_dir)
    if missing or unexpected:
        print(f"  [warn] load_state_dict(strict=False)")
        if missing:    print(f"    missing keys: {len(missing)}")
        if unexpected: print(f"    unexpected keys: {len(unexpected)}")
    print(f"  loaded: {wpath}")

    model.eval()

    # Full-bag val evaluation
    # If you use my thesis version:
    val_acc, val_loss, df_all, df_thesis = evaluate_full_bag_on_split_thesis(
        val_bags, model=model, processor=processor, device=device, chunk_size=CHUNK_SIZE
    )

    print(f"  Full-bag val accuracy: {val_acc:.4f}, loss: {val_loss:.4f}")

    val_results_per_ckpt[ckpt_dir.name] = {
        "step": ckpt_step(ckpt_dir),
        "accuracy": float(val_acc),
        "loss": float(val_loss),
        "weights_path": wpath,
    }

    # Selection logic with epsilon:
    # 1) higher accuracy wins
    # 2) if accuracy is within EPS, lower loss wins
    acc_better = (val_acc > best_acc + EPS)
    acc_tied   = (abs(val_acc - best_acc) <= EPS)
    loss_better = (val_loss < best_loss - EPS)

    is_better = acc_better or (acc_tied and loss_better)

    if is_better:
        best_acc = float(val_acc)
        best_loss = float(val_loss)
        best_ckpt = ckpt_dir
        best_val_df_raw = df_all
        best_val_df_thesis = df_thesis  # if using thesis version

print("\nBest checkpoint by full-bag val accuracy+loss:", best_ckpt)
print(f"Best full-bag val accuracy: {best_acc:.4f}, loss: {best_loss:.4f}")

# ----------------------------
# Save summary JSON
# ----------------------------
summary = {
    "model_name": model_name,
    "chunk_size": CHUNK_SIZE,
    "n_val_pages": int(len(val_bags)),
    "selection_rule": "max accuracy; tie (±EPS) -> min loss",
    "eps": EPS,
    "per_checkpoint": val_results_per_ckpt,
    "best_checkpoint": best_ckpt.name if best_ckpt is not None else None,
    "best_fullbag_val_accuracy": float(best_acc),
    "best_fullbag_val_loss": float(best_loss),
}

with open(RESULTS_DIR / "fullbag_val_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

# ----------------------------
# Optional: save the best checkpoint val table for thesis
# ----------------------------
if best_val_df_raw is not None:
    out_csv = RESULTS_DIR / "best_fullbag_val_predictions.csv"
    best_val_df_raw.to_csv(out_csv, index=False)
    print("Saved:", out_csv)

    # If you switch to thesis df version, also save LaTeX:
    out_tex = RESULTS_DIR / "best_fullbag_val_table.tex"
    best_val_df_thesis.to_latex(out_tex, index=False, float_format="%.3f")
    print("Saved:", out_tex)

Found checkpoints: ['checkpoint-101', 'checkpoint-202', 'checkpoint-303', 'checkpoint-404', 'checkpoint-505', 'checkpoint-606']

Evaluating full-bag val on checkpoint-101 ...


Some weights of ViTModel were not initialized from the model checkpoint at facebook/deit-small-patch16-224 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  loaded: mil_vit_results/checkpoint-101/model.safetensors


Full-bag eval: 100%|██████████| 16/16 [00:30<00:00,  1.91s/it]


  Full-bag val accuracy: 0.5625, loss: 0.5875

Evaluating full-bag val on checkpoint-202 ...


Some weights of ViTModel were not initialized from the model checkpoint at facebook/deit-small-patch16-224 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


... (15 outputs left out)

Some weights of ViTModel were not initialized from the model checkpoint at facebook/deit-small-patch16-224 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  loaded: mil_vit_results/checkpoint-606/model.safetensors


Full-bag eval: 100%|██████████| 16/16 [00:30<00:00,  1.88s/it]

  Full-bag val accuracy: 0.9375, loss: 0.2123

Best checkpoint by full-bag val accuracy+loss: mil_vit_results/checkpoint-606
Best full-bag val accuracy: 0.9375, loss: 0.2123
Saved: mil_vit_results/best_fullbag_val_predictions.csv
Saved: mil_vit_results/best_fullbag_val_table.tex


## Predict on test set 

In [13]:

CHUNK_SIZE = 256

# Rebuild and load best model
best_model = VisionTransformerMILModel(
    model_name=model_name,
    num_labels=2,
    mil_dropout=0.3,
).to(device)

# Load best checkpoint (safetensors)
best_safetensors = best_ckpt / "model.safetensors"
best_state_dict = load_file(best_safetensors)

missing, unexpected = best_model.load_state_dict(best_state_dict, strict=False)
if missing or unexpected:
    print("[warn] load_state_dict(strict=False)")
    if missing:    print(f"  missing keys: {len(missing)}")
    if unexpected: print(f"  unexpected keys: {len(unexpected)}")

best_model.eval()

# ----------------------------
# Full-bag TEST eval (thesis version)
# ----------------------------
test_acc, test_loss, test_df_all, test_df_thesis = evaluate_full_bag_on_split_thesis(
    test_bags,
    model=best_model,
    processor=processor,
    device=device,
    chunk_size=CHUNK_SIZE,
    label_map={0: "French", 1: "German"},   # adjust capitalization to your thesis
    round_probs=3,
)

print(f"Full-bag TEST accuracy (page-level): {test_acc:.4f}")
print(f"Full-bag TEST loss (page-level CE):  {test_loss:.4f}")

# ----------------------------
# Save artifacts to RESULTS_DIR
# ----------------------------
# 1) Full debug predictions
out_all_csv = RESULTS_DIR / "test_predictions_mil_fullbag_bestckpt_all.csv"
test_df_all.to_csv(out_all_csv, index=False)

# 2) Thesis-ready table (clean)
out_thesis_csv = RESULTS_DIR / "test_predictions_mil_fullbag_bestckpt_thesis.csv"
test_df_thesis.to_csv(out_thesis_csv, index=False)

out_thesis_tex = RESULTS_DIR / "test_predictions_mil_fullbag_bestckpt_thesis.tex"
test_df_thesis.to_latex(out_thesis_tex, index=False, float_format="%.3f")

# 3) Metrics JSON
out_metrics = RESULTS_DIR / "test_metrics_mil_fullbag_bestckpt.json"
with open(out_metrics, "w") as f:
    json.dump(
        {
            "model_name": model_name,
            "best_checkpoint": best_ckpt.name,
            "best_checkpoint_path": str(best_ckpt),
            "weights_file": str(best_safetensors),
            "chunk_size": CHUNK_SIZE,
            "n_test_pages": int(len(test_bags)),
            "full_bag_test_accuracy": float(test_acc),
            "full_bag_test_loss": float(test_loss),
        },
        f,
        indent=2,
    )

print("Saved:")
print(" ", out_all_csv)
print(" ", out_thesis_csv)
print(" ", out_thesis_tex)
print(" ", out_metrics)

Some weights of ViTModel were not initialized from the model checkpoint at facebook/deit-small-patch16-224 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Full-bag eval: 100%|██████████| 25/25 [00:09<00:00,  2.68it/s]

Full-bag TEST accuracy (page-level): 0.8000
Full-bag TEST loss (page-level CE):  0.7663
Saved:
  mil_vit_results/test_predictions_mil_fullbag_bestckpt_all.csv
  mil_vit_results/test_predictions_mil_fullbag_bestckpt_thesis.csv
  mil_vit_results/test_predictions_mil_fullbag_bestckpt_thesis.tex
  mil_vit_results/test_metrics_mil_fullbag_bestckpt.json


## Visualize Attention Overlays

In [14]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Extract x, y pixel coordinates from filename
_xy_pattern = re.compile(r"_y(\d+)_x(\d+)_")

def extract_xy_from_path(path_str: str):
    """
    Parse patch coordinates from filename convention:
    026_003_00_y03808_x01344_0125.png  --> (x=1344, y=3808)
    """
    fname = Path(path_str).name
    m = _xy_pattern.search(fname)
    if m is None:
        raise ValueError(f"Cannot parse y/x from filename: {fname}")
    y = int(m.group(1))
    x = int(m.group(2))
    return x, y  # (x, y)


@torch.no_grad()
def compute_bag_attention(bag, model, processor, device, chunk_size=256):
    """
    Full-bag forward pass with:
        - logits (num_labels)
        - attention weights (N_patches,)
        - coordinates list (N_patches * (x,y))
    """
    paths = bag["patch_paths"]
    all_cls_cpu = []
    coords = []

    model.to(device)
    model.eval()

    for start in range(0, len(paths), chunk_size):
        chunk_paths = paths[start : start + chunk_size]
        images = [Image.open(p).convert("RGB") for p in chunk_paths]

        coords.extend([extract_xy_from_path(p) for p in chunk_paths])

        inputs = processor(images=images, return_tensors="pt")
        pixel_values = inputs["pixel_values"].to(device)

        outputs = model.backbone(pixel_values=pixel_values)
        cls_emb = outputs.last_hidden_state[:, 0, :]  # CLS token

        all_cls_cpu.append(cls_emb.cpu())

        del pixel_values, outputs, cls_emb
        torch.cuda.empty_cache()

    H = torch.cat(all_cls_cpu, dim=0).to(device)  # (N_patches, d)

    logits, att = model.mil_head(H)
    logits = logits.squeeze(0).cpu().numpy()
    att = att.squeeze(1).cpu().numpy()

    # Normalize attention to [0,1]
    att = att / (att.max() + 1e-8)

    return logits, att, coords


def build_attention_heatmap(att, coords, patch_size=224, downsample=4):
    """
    Build a 2D heatmap (downsampled) where each patch contributes its attention weight.
    """
    xs, ys = zip(*coords)
    max_x = max(xs) + patch_size
    max_y = max(ys) + patch_size

    H = max_y // downsample + 1
    W = max_x // downsample + 1

    heat = np.zeros((H, W), dtype=np.float32)

    for (x, y), a in zip(coords, att):
        y0 = y // downsample
        y1 = (y + patch_size) // downsample
        x0 = x // downsample
        x1 = (x + patch_size) // downsample
        heat[y0:y1, x0:x1] += a

    if heat.max() > 0:
        heat = heat / heat.max()

    return heat


def save_heatmap_png(heat, out_path, title="MIL Attention Heatmap"):
    plt.figure(figsize=(6, 8))
    plt.imshow(heat, cmap="inferno")
    plt.colorbar(label="attention")
    plt.title(title)
    plt.axis("off")
    plt.savefig(out_path, bbox_inches="tight", dpi=200)
    plt.close()


def overlay_attention_on_page(page_image_path, heat, out_path, alpha=0.4):
    """
    Overlay heatmap on top of the original page image.
    """
    page_img = Image.open(page_image_path).convert("RGB")

    # Resize image to heatmap resolution
    h, w = heat.shape
    page_resized = page_img.resize((w, h))

    plt.figure(figsize=(6, 8))
    plt.imshow(page_resized)
    plt.imshow(heat, cmap="jet", alpha=alpha)
    plt.axis("off")
    plt.title("MIL Attention Overlay")
    plt.savefig(out_path, bbox_inches="tight", dpi=200)
    plt.close()

In [15]:

def extract_region_key_from_patch_path(p: str) -> str:
    """
    From patch path like:
      026_003_00_y03808_x01344_0125.png
    return:
      "026_003_00"  (ms_id_page_id_region_id)
    """
    stem = Path(p).stem
    parts = stem.split("_")
    if len(parts) < 3:
        raise ValueError(f"Unexpected patch filename format: {Path(p).name}")
    return f"{parts[0]}_{parts[1]}_{parts[2]}"


def find_region_image(full_region_image_dir: str, region_key: str):
    """
    Search for a region image file like:
      {region_key}.png / .jpg / .jpeg / .tif / .tiff

    Returns Path or None.
    """
    full_region_image_dir = Path(full_region_image_dir)

    # Common extensions; add more if you need
    exts = ["png", "jpg", "jpeg", "tif", "tiff", "bmp"]
    for ext in exts:
        candidate = full_region_image_dir / f"{region_key}.{ext}"
        if candidate.exists():
            return candidate

    # Fallback: glob any extension
    matches = list(full_region_image_dir.glob(f"{region_key}.*"))
    return matches[0] if matches else None


In [16]:

def visualize_bag_attention_on_region(
    bag,
    model,
    processor,
    RESULTS_DIR,
    full_region_image_dir,     # REQUIRED for region overlay
    patch_size=224,
    downsample=4,
):
    """
    Produces and saves:
        - heatmap.png   (attention heatmap)
        - overlay.png   (heatmap over the text-region image)
        - metadata.json (logits, predicted class, etc.)

    Assumes region images are named:
        {ms_id}_{page_id}_{region_id}.ext
    """
    # Determine region key from first patch
    first_patch = bag["patch_paths"][0]
    region_key = extract_region_key_from_patch_path(first_patch)

    out_dir = Path(RESULTS_DIR) / "attention_visualizations" / region_key
    out_dir.mkdir(parents=True, exist_ok=True)

    # 1) Compute logits + att
    logits, att, coords = compute_bag_attention(
        bag, model=model, processor=processor, device=device
    )
    pred = int(logits.argmax())

    # 2) Heatmap
    heat = build_attention_heatmap(
        att, coords, patch_size=patch_size, downsample=downsample
    )

    # 3) Save heatmap
    heatmap_path = out_dir / "heatmap.png"
    save_heatmap_png(heat, heatmap_path, title=f"Attention Heatmap ({region_key})")

    # 4) Overlay on region image
    overlay_path = out_dir / "overlay.png"
    region_img_path = find_region_image(full_region_image_dir, region_key)
    if region_img_path is None:
        print(f"[warning] Cannot find region image for key={region_key} in {full_region_image_dir}")
    else:
        overlay_attention_on_page(region_img_path, heat, overlay_path, alpha=0.45)

    # 5) Metadata
    with open(out_dir / "metadata.json", "w") as f:
        json.dump(
            {
                "region_key": region_key,
                "true_label": int(bag["label"]),
                "logits": logits.tolist(),
                "predicted_class": pred,
                "num_patches": int(len(att)),
                "region_image_path": str(region_img_path) if region_img_path else None,
                "downsample": int(downsample),
                "patch_size": int(patch_size),
            },
            f,
            indent=2,
        )

    print(f"[OK] Saved region visualizations to: {out_dir}")
    return out_dir


In [41]:
# Bag indices to visualize
bag_indexes_to_visualize = [
    1,    # ms 009, page 000
    19,    # ms 021, page 000
    23, # ms 025, page 000
]
for i in bag_indexes_to_visualize:
    bag = test_bags[i]
    full_region_image_dir = "ASC_dataset_v3/test/"
    if i == 1:
        full_region_image_dir += "french"
    else:
        full_region_image_dir += "german"

    visualize_bag_attention_on_region(
        bag,
        model=best_model,
        processor=processor,
        RESULTS_DIR=RESULTS_DIR,
        full_region_image_dir=full_region_image_dir,
        patch_size=224,
        downsample=4,
    )


[OK] Saved region visualizations to: mil_vit_results/attention_visualizations/009_000_00
[OK] Saved region visualizations to: mil_vit_results/attention_visualizations/021_000_00
[OK] Saved region visualizations to: mil_vit_results/attention_visualizations/025_000_00
